## Article 9: Deep Learning - Analysis Notebook

This notebook reads benchmark outputs from `results/data/article_09_benchmarks.json`
and related JSON files. If those files are missing, the notebook **fails fast** with
`FileNotFoundError` rather than fabricating numbers. Generate the JSON first via:

```bash
uv run python benchmarks/run_article_09.py
```


In [1]:
"""Article 9: Deep Learning - Analysis Notebook.

Reads benchmark JSON outputs and generates summary charts.
Hard-fails with FileNotFoundError if the underlying JSON is missing -
run benchmarks/run_article_09.py first to generate the data.
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
# WHY Agg backend: this notebook is executed headlessly via nbconvert
# (uv run jupyter nbconvert --execute ...). Agg renders charts to PNG files
# without requiring a display server, making it CI-safe. Interactive backends
# (TkAgg, MacOSX) would raise errors in headless environments.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
DATA = ROOT / "results" / "data"
CHARTS = ROOT / "results" / "charts" / "article_09"
CHARTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})
print("Setup complete. Charts will be written to:", CHARTS)

Setup complete. Charts will be written to: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_09


In [2]:
# Cell 2: Training Recall@5 chart
# WHY: Validation Recall@5 before vs after fine-tuning is the headline metric
# for contrastive domain adaptation. The training script records aggregate
# stats (start/end Recall@5) only, so we plot the two honest endpoints rather
# than fabricate an epoch-by-epoch curve.

training_json = ROOT / "models" / "bge_finetuned" / "training_history.json"

if not training_json.exists():
    raise FileNotFoundError(
        f"Required benchmark output missing: {training_json}. "
        "Run `uv run python benchmarks/run_article_09.py` first."
    )
history = json.loads(training_json.read_text())

print(f"Model:              {history['model']}")
print(f"Device:             {history['device']}")
print(f"Epochs:             {history['epochs']}")
print(f"Baseline Recall@5:  {history['baseline_recall_at_5']:.4f}")
print(f"Final Recall@5:     {history['final_recall_at_5']:.4f}")
print(f"Improvement:        {history['improvement']:+.4f}")
print(f"Training time:      {history['training_seconds']:.0f}s")

epochs_x = list(range(1, history["epochs"] + 1))
# Linear interpolation between real start/end Recall@5 values. With the
# typical 1-2 epoch runs this is just the two endpoints; longer runs would
# need per-epoch checkpoint logging in the training script for an honest
# mid-epoch curve.
recall_curve = list(np.linspace(history["baseline_recall_at_5"], history["final_recall_at_5"], history["epochs"]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(epochs_x, recall_curve, marker="o", color="#e63946", linewidth=2, label="Val Recall@5")
ax.axhline(history["baseline_recall_at_5"], linestyle="--", color="#adb5bd", label="Stock baseline")
ax.set_xlabel("Epoch")
ax.set_ylabel("Recall@5")
ax.set_title(f"Fine-tuning: {history['model']} ({history['epochs']} epoch(s))")
ax.set_xticks(epochs_x)
ax.set_ylim(0, 1)
ax.legend()

out = CHARTS / "06_training_curves.png"
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

Model:              BAAI/bge-base-en-v1.5
Device:             mps
Epochs:             2
Baseline Recall@5:  0.6125
Final Recall@5:     0.7825
Improvement:        +0.1700
Training time:      359s
Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_09/06_training_curves.png


In [3]:
# Cell 3: Recall@K bar chart
# WHY: Recall@K on the full 200-doc corpus is the production-relevant metric.
# Training-set Recall@5 only tests 400 pairs; corpus Recall@K reveals whether
# the model generalises to unseen documents - the harder, more realistic test.

bench_json = DATA / "article_09_benchmarks.json"

if not bench_json.exists():
    raise FileNotFoundError(
        f"Required benchmark output missing: {bench_json}. "
        "Run `uv run python benchmarks/run_article_09.py` first."
    )
bench = json.loads(bench_json.read_text())

ks = [1, 3, 5]
stock_vals = [bench[f"recall_at_{k}"]["stock"] for k in ks]
ft_vals = [bench[f"recall_at_{k}"]["finetuned"] for k in ks]

x = np.arange(len(ks))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width / 2, stock_vals, width, label="Stock BGE-base-en-v1.5", color="#457b9d")
ax.bar(x + width / 2, ft_vals, width, label="Fine-tuned (domain)", color="#e63946")
ax.set_xlabel("K")
ax.set_ylabel("Recall@K")
ax.set_title("Stock vs Fine-tuned: Recall@K on Full Corpus")
ax.set_xticks(x)
ax.set_xticklabels([f"K={k}" for k in ks])
ax.set_ylim(0, 1)
ax.legend()
out = CHARTS / "07_recall_at_k.png"
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")
print(f"Latency - Stock: {bench['latency_ms']['stock_median']:.1f}ms  Fine-tuned: {bench['latency_ms']['finetuned_median']:.1f}ms")

Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_09/07_recall_at_k.png
Latency - Stock: 7.2ms  Fine-tuned: 6.7ms


In [4]:
# Cell 4: PyTorch inference optimisation chart
# WHY: torch.compile() and INT8 quantisation are commonly cited as cheap
# production wins. Measured on this CPU setup the picture is the opposite:
# compile is ~0.92x (slower than eager) and INT8 is ~0.48x (half the speed)
# while preserving cosine similarity ~0.98 and shrinking the model 4x. The
# chart documents the honest CPU result; both optimisations land where
# they actually pay off (compile on CUDA/MPS once warm; INT8 on
# memory-constrained serving where the size win matters more than wall time).

opt_json = DATA / "pytorch_optimizations.json"

if not opt_json.exists():
    raise FileNotFoundError(
        f"Required benchmark output missing: {opt_json}. "
        "Run `uv run python benchmarks/run_article_09.py` first."
    )
opt = json.loads(opt_json.read_text())

quant = opt["quantization"]
fp32_size = quant["fp32_size_mb"]
int8_size = quant["int8_size_mb"]

eager_ms = opt["compile"]["eager_ms"]
labels = ["Eager", "torch.compile", "INT8 Quant"]
latencies = [eager_ms, opt["compile"]["compiled_ms"], quant["int8_ms"]]
colors = ["#adb5bd", "#457b9d", "#e63946"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, latencies, color=colors, width=0.5)
ax.set_ylabel("Median latency (ms, single query)")
ax.set_title("PyTorch Inference Optimisations (CPU)")
for bar, val in zip(bars, latencies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{val:.1f}ms", ha="center", fontsize=9)
out = CHARTS / "08_pytorch_speedup.png"
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")
print(f"compile speedup: {opt['compile']['speedup']:.2f}x  INT8 speedup: {quant['speedup']:.2f}x")
print(f"Model size: {fp32_size:.0f}MB -> {int8_size:.0f}MB (INT8, cosine={quant['cosine_similarity']:.3f})")

Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_09/08_pytorch_speedup.png
compile speedup: 0.92x  INT8 speedup: 0.48x
Model size: 438MB -> 110MB (INT8, cosine=0.983)


In [5]:
# Cell 5: PyTorch vs JAX latency comparison
# WHY: The folk wisdom is that JAX's jit+vmap+XLA stack outperforms PyTorch
# eager at large batch sizes. Measured on CPU here, that does NOT hold for
# forward ops: PyTorch wins matmul and softmax at every tested batch size
# (64-1024 for matmul, 128-8192 for softmax). JAX wins gradient ops at
# every tested size (32-256). The honest reading is "JAX wins on autodiff
# pipelines on CPU; PyTorch wins on forward inference on CPU; the XLA win
# happens on TPU/large GPU clusters, not in this single-host setup."

jax_json = DATA / "pytorch_vs_jax_benchmark.json"

if not jax_json.exists():
    raise FileNotFoundError(
        f"Required benchmark output missing: {jax_json}. "
        "Run `uv run python benchmarks/run_article_09.py` first."
    )
jax_data = json.loads(jax_json.read_text())

ops = ["matmul", "softmax", "grad"]
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("PyTorch vs JAX: Operation Latency by Batch Size (CPU)", fontsize=13)

for ax, op in zip(axes, ops):
    sizes = sorted(jax_data["pytorch"][op].keys(), key=int)
    pt_vals = [jax_data["pytorch"][op][s] for s in sizes]
    jax_vals = [jax_data["jax"][op][s] for s in sizes]
    x = np.arange(len(sizes))
    width = 0.35
    ax.bar(x - width / 2, pt_vals, width, label="PyTorch", color="#457b9d")
    ax.bar(x + width / 2, jax_vals, width, label="JAX", color="#e63946")
    ax.set_title(op)
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Latency (ms)")
    ax.set_xticks(x)
    ax.set_xticklabels(sizes)
    ax.legend(fontsize=8)

out = CHARTS / "09_pytorch_vs_jax.png"
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_09/09_pytorch_vs_jax.png


## Article 9: Deep Learning - Measured Findings

All numbers below are read directly from the JSON files written by `benchmarks/run_article_09.py` and `examples/article_09_dl/custom_reranker.py --train --eval`. The article around this notebook reads the same files and is not allowed to override them.

### Headline results

| Finding | Measured |
|---------|----------|
| BGE fine-tune on the curated `(query, expected_answer)` validation split | Recall@5 0.6125 -> 0.7825 (+17.0pp, training-only metric) |
| BGE fine-tune on the full 207-doc retrieval corpus | Recall@1 -6.78pp; Recall@3 -8.47pp; Recall@5 **-10.73pp** (regression) |
| BGE encode latency (single query) | Stock 7.18 ms vs fine-tuned 6.66 ms (architecture unchanged) |
| `torch.compile()` on CPU vs eager FP32 | 23.97 -> 26.12 ms (**0.92x, slower**) |
| INT8 dynamic quantisation on CPU vs FP32 | 23.49 -> 48.95 ms (**0.48x, slower**); model 437.9 MB -> 109.5 MB (4.0x); cosine vs FP32 0.983 |
| PyTorch vs JAX (CPU) | PyTorch wins matmul + softmax at every tested batch size; JAX wins gradient ops at every tested size |
| Custom cross-encoder reranker vs FlashRank (39 queries) | NDCG@5 0.874 vs 0.7606 (+11.3pp); latency 121 ms vs 341 ms (~2.8x faster) |

### What this is honestly teaching

1. **Validation Recall != deployment Recall.** Training on `(query, expected_answer)` pairs lifted validation Recall@5 by 17 pp and *regressed* full-corpus Recall@5 by ~11 pp. The model learned `query -> short summary` alignment, not `query -> long document` alignment. Fix is to re-train on `(query, doc-chunk)` pairs (deferred follow-up).
2. **Compile and INT8 are not free wins on CPU.** Both increased single-query wall time on this M4 host. INT8 still pays for itself when the constraint is RAM/disk (4x size reduction, 0.98 cosine fidelity). torch.compile pays for itself on CUDA/MPS once the warmup is amortised, not on CPU eager workloads.
3. **JAX is not "always faster at scale".** On this single-host CPU run, PyTorch wins forward inference (matmul, softmax) across every tested batch size; JAX wins backward (gradient) ops across every tested size. The XLA scaling story is a TPU/multi-host-GPU claim, not a CPU claim.
4. **A fine-tuned cross-encoder beats a comparable FlashRank baseline on both NDCG and latency** at this corpus size (39-query eval). This is the cleanest "fine-tune actually paid off" result in the article.

### Charts generated by this notebook

- `06_training_curves.png` - Validation Recall@5 endpoints (baseline vs final epoch)
- `07_recall_at_k.png` - Stock vs fine-tuned Recall@K on the full retrieval corpus (the regression chart)
- `08_pytorch_speedup.png` - Eager vs torch.compile vs INT8 single-query latency (CPU)
- `09_pytorch_vs_jax.png` - Per-op latency by batch size (matmul, softmax, grad), CPU

Reranker chart `04_reranker_comparison.png` is written by `custom_reranker.py` itself, not by this notebook.